In [ ]:
#--------------------------------------------------
# Step 1: Imports & GPU Check
#--------------------------------------------------

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Enable GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
#--------------------------------------------------
# Step 2: Custom PyTorch Dataset Class
#--------------------------------------------------

In [3]:
# The 20 official Pascal VOC classes
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

class PascalVOCMultiLabel(Dataset):
    def __init__(self, root, image_set='train', transform=None):
        # Initialize built-in Pascal VOC dataset downloader
        self.voc = datasets.VOCDetection(root=root, year='2012', image_set=image_set, download=True)
        self.transform = transform
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(VOC_CLASSES)}

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        img, target = self.voc[idx]

        # Create a zero-filled vector for 20 classes: e.g., [0, 0, 0, ..., 0]
        label_vector = torch.zeros(len(VOC_CLASSES), dtype=torch.float32)

        # Parse XML annotations to extract present object classes
        objects = target['annotation']['object']
        if not isinstance(objects, list):
            objects = [objects]

        for obj in objects:
            cls_name = obj['name']
            if cls_name in self.class_to_idx:
                # Set index to 1.0 if class exists in the image
                label_vector[self.class_to_idx[cls_name]] = 1.0

        if self.transform:
            img = self.transform(img)

        return img, label_vector

In [ ]:
#--------------------------------------------------
# Step 3: Transformations & DataLoaders
#--------------------------------------------------

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

# Create Train and Validation Datasets
train_dataset = PascalVOCMultiLabel(root='./data', image_set='train', transform=data_transforms['train'])
val_dataset = PascalVOCMultiLabel(root='./data', image_set='val', transform=data_transforms['val'])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

100%|██████████| 2.00G/2.00G [00:52<00:00, 37.9MB/s]


In [ ]:
#--------------------------------------------------
# Step 4: Model Architecture, Loss Function & Optimizer
#--------------------------------------------------

In [5]:
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze feature extractor layers
for param in model.parameters():
    param.requires_grad = False

# Replace output layer to output 20 class logits
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(VOC_CLASSES))
model = model.to(device)

# --- CRITICAL DIFFERENCE FOR MULTI-LABEL ---
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 107MB/s]


In [ ]:
#--------------------------------------------------
# Step 5: Training and Validation Loop
#--------------------------------------------------

In [ ]:
epochs = 5
threshold = 0.5  # Probability threshold to consider a class present

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)          # Shape: (batch_size, 20)
        loss = criterion(outputs, labels)# Computes BCE loss across all 20 classes
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

    val_loss = val_loss / len(val_dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
#--------------------------------------------------
# Step 5: Training and Validation Loop
#--------------------------------------------------

In [ ]:
epochs = 5
threshold = 0.5  # Probability threshold to consider a class present

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)          # Shape: (batch_size, 20)
        loss = criterion(outputs, labels)# Computes BCE loss across all 20 classes
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

    val_loss = val_loss / len(val_dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
#--------------------------------------------------
# Step 6: Prediction on Uploaded Images
#--------------------------------------------------

In [ ]:
from google.colab import files
import io

print("Upload an image containing multiple objects:")
uploaded = files.upload()

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

for filename in uploaded.keys():
    raw_image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
    input_tensor = inference_transform(raw_image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        # Apply Sigmoid to convert raw logits to independent probabilities (0 to 1)
        probabilities = torch.sigmoid(logits)[0]

    # Extract all classes that cross the 50% probability threshold
    detected_classes = []
    for idx, prob in enumerate(probabilities):
        if prob.item() >= 0.5:
            detected_classes.append(f"{VOC_CLASSES[idx]} ({prob.item()*100:.1f}%)")

    # Display Result
    plt.figure(figsize=(6, 6))
    plt.imshow(raw_image)
    title_str = "Detected: " + ", ".join(detected_classes) if detected_classes else "No classes detected (>50%)"
    plt.title(title_str, fontsize=12, color="blue")
    plt.axis("off")
    plt.show()